In [1]:
import pandas as pd

df = pd.read_csv("C:\\Users\\POL\\Downloads\\comisiones_netas.csv", sep=";", encoding="latin-1")
df

,"trimestre,comisiones_netas,fuente"
0,"2015Q1,513.0,CaixaBank1Q16results_Excel_SPA.XLSX"
1,"2015Q2,538.0,CaixaBank2Q16results_Excel_SPA.XLSX"
2,"2015Q3,524.0,CaixaBank3Q16results_Excel_SPA.XLSX"
3,"2015Q4,515.0,CaixaBankFY16results_Excel_SPA.XLSX"
4,"2016Q1,488.0,CaixaBank1Q17results_Excel_SPA.XLSX"
5,"2016Q2,522.0,CaixaBank1Q17results_Excel_SPA.XLSX"
6,"2016Q3,536.0,CaixaBank1Q17results_Excel_SPA.XLSX"
7,"2016Q4,544.0,CaixaBank1Q17results_Excel_SPA.XLSX"
8,"2017Q1,588.0,CaixaBank1Q18results_Excel_SPA.XLSX"
9,"2017Q2,664.0,CaixaBank2Q18results_Excel_SPA.XLSX"


In [ ]:
import pandas as pd

df = pd.read_csv(r"C:\Users\POL\Downloads\comisiones_netas.csv")
df["trimestre"] = pd.PeriodIndex(df["trimestre"], freq="Q")
df = df.set_index("trimestre")
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

df = pd.read_csv(r"C:\Users\POL\Downloads\comisiones_netas.csv")
df["trimestre"] = pd.PeriodIndex(df["trimestre"], freq="Q")
s = df.set_index("trimestre")["comisiones_netas"]["2016Q1":]
s.index = s.index.to_timestamp()

fig, ax = plt.subplots(figsize=(10, 4))
s.plot(ax=ax, marker="o")
for fecha, texto in [("2017-01-01", "BPI"), ("2021-01-01", "Bankia")]:
    ax.axvline(pd.Timestamp(fecha), ls="--", color="gray")
    ax.text(pd.Timestamp(fecha), ax.get_ylim()[1], texto, va="top")
ax.set_title("Comisiones netas CaixaBank (M€, trimestral)")
plt.show()

seasonal_decompose(s, model="multiplicative", period=4).plot()
plt.show()

In [6]:
import pandas as pd

df = pd.read_csv(r"C:\Users\POL\Downloads\comisiones_netas.csv")
df["trimestre"] = pd.PeriodIndex(df["trimestre"], freq="Q")
s = df.set_index("trimestre")["comisiones_netas"]["2016Q1":]



In [7]:
# Variable exógena: 0 hasta 2021T1, 1 desde 2021T2
exog = pd.DataFrame(index=s.index)
exog["bankia"] = (s.index >= pd.Period("2021Q2", freq="Q")).astype(int)



In [ ]:
# Entrenamiento y prueba (últimos 4 trimestres)
n_test = 4
y_train, y_test = s.iloc[:-n_test], s.iloc[-n_test:]
x_train, x_test = exog.iloc[:-n_test], exog.iloc[-n_test:]

print(y_train.index[-1], list(y_test.index.astype(str)))
print(exog["bankia"].value_counts())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ---------- 1) Cargar la serie (2016T1 en adelante) ----------
RUTA = r"C:\Users\POL\Downloads\comisiones_netas.csv"
df = pd.read_csv(RUTA)
df["trimestre"] = pd.PeriodIndex(df["trimestre"], freq="Q")
s = df.set_index("trimestre")["comisiones_netas"]["2016Q1":]

# ---------- 2) Ajuste por Bankia: subimos el histórico anterior a 2021T2 ----------
antes = s["2020Q2":"2021Q1"].mean()      # 4 trimestres antes de la fusión
despues = s["2021Q2":"2022Q1"].mean()    # 4 trimestres después
K = despues / antes
print(f"Factor Bankia estimado: {K:.3f}")
# K = 1.45   # <- descomenta y cambia el número si quieres fijar el factor a mano

s_adj = s.copy()
s_adj.loc[s_adj.index < pd.Period("2021Q2", freq="Q")] *= K

# ---------- 3) Entrenamiento y prueba (últimos 4 trimestres, valores reales) ----------
n_test = 4
y_train, y_test = s_adj.iloc[:-n_test], s_adj.iloc[-n_test:]

def metricas(real, pred):
    real, pred = np.asarray(real), np.asarray(pred)
    mae = np.mean(np.abs(real - pred))
    mape = np.mean(np.abs(real - pred) / np.abs(real)) * 100
    return round(float(mae), 1), round(float(mape), 2)

# ---------- 4) Modelos ----------
config = {
    "HW tendencia amortiguada": dict(trend="add", damped_trend=True),
    "HW sin tendencia": dict(trend=None, damped_trend=False),
}
preds = {"Naive estacional": s_adj.shift(4).loc[y_test.index].values}
for nombre, kw in config.items():
    m = ExponentialSmoothing(y_train.values, seasonal="add", seasonal_periods=4, **kw).fit()
    preds[nombre] = m.forecast(n_test)

tabla = pd.DataFrame(
    {nombre: metricas(y_test, p) for nombre, p in preds.items()},
    index=["MAE (M€)", "MAPE (%)"],
).T
print(tabla)
mejor = tabla.loc[list(config), "MAE (M€)"].idxmin()
print("Modelo elegido:", mejor)

# ---------- 5) Predicción final con todos los datos ----------
final = ExponentialSmoothing(
    s_adj.values, seasonal="add", seasonal_periods=4, **config[mejor]
).fit()
H = 8
pred = final.forecast(H)
sigma = np.std(final.resid, ddof=1)
futuro = pd.period_range(s_adj.index[-1] + 1, periods=H, freq="Q")
salida = pd.DataFrame(
    {"prevision": pred, "minimo": pred - 1.96 * sigma, "maximo": pred + 1.96 * sigma},
    index=futuro.astype(str),
).round(1)
print(salida)

# ---------- 6) Gráfico ----------
fig, ax = plt.subplots(figsize=(10, 4))
hist = s_adj.copy(); hist.index = hist.index.to_timestamp()
ax.plot(hist.index, hist.values, marker="o", label="Histórico (ajustado por Bankia)")
fx = futuro.to_timestamp()
ax.plot(fx, pred, marker="o", color="tab:red", label="Previsión")
ax.fill_between(fx, pred - 1.96 * sigma, pred + 1.96 * sigma, color="tab:red", alpha=0.15, label="Rango aprox. 95 %")
ax.axvline(pd.Timestamp("2021-04-01"), ls="--", color="gray")
ax.set_title("Comisiones netas CaixaBank (M€, trimestral)")
ax.legend()
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
seasonal_decompose(hist, model="multiplicative", period=4).plot()
plt.show()